# RVSH bot on Google Colab

Runs the bot in this notebook — good for **trying it or a change**; Colab ends free sessions after some hours or when
the tab is idle, so keep production on a server (`docs/SETUP.md`: Docker / systemd).

1. **🔑 Secrets** (left bar, key icon) → add, and switch on *Notebook access* for each:
   `BOT_TOKEN`, `API_ID`, `API_HASH`, `OWNER_ID`, `DUMP_CHANNEL_ID`, `IMAGE_DUMP_CHANNEL_ID`, `BRANDING`,
   `IMAGE_BOT_TOKENS`, `MONGO_URL` (**needed here** — Colab's disk is wiped when the session ends).
   Optional: `QUIZ_DUMP_CHANNEL_ID`, `FORCE_SUB_CHANNELS`, `HTTP_PROXY`, `CLOUDFLARED_TOKEN`, `WEBAPP_BASE_URL`,
   `GITHUB_TOKEN` (for a private repo).
2. Fill the cell below (where the code comes from), then **Runtime → Run all**.
3. Only **one copy** of the bot may run: a copy started here asks the running one (PC / server) to hand over and that
   one stops. Stop this one with ⏹ (the bot stops gracefully) before starting the PC / server copy again.

In [ ]:
# Where the code comes from — one of the two:
REPO_URL = ""          # e.g. "https://github.com/you/rvsh.git" (a private repo: add the GITHUB_TOKEN secret)
BRANCH = "main"
ZIP_PATH = ""          # or upload the deploy zip (dist/rvsh-bot-<date>.zip) with the 📁 Files panel and put its path here
USE_DRIVE = False      # True: keep downloads / logs in Google Drive (MyDrive/rvsh-data) across sessions

In [ ]:
import os, shutil, subprocess, sys, zipfile

CODE_DIR = "/content/rvsh"
DATA = "/content/rvsh-data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/rvsh-data"
os.makedirs(DATA, exist_ok=True)

shutil.rmtree(CODE_DIR, ignore_errors=True)
if REPO_URL:
    url = REPO_URL
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok and url.startswith("https://"):
            url = url.replace("https://", f"https://x-access-token:{tok}@", 1)
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, CODE_DIR], check=True)
elif ZIP_PATH:
    tmp = "/content/_rvsh_unzip"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(tmp)
    inner = [d for d in os.listdir(tmp) if os.path.isdir(os.path.join(tmp, d))]
    src = os.path.join(tmp, inner[0]) if len(inner) == 1 and not os.path.exists(os.path.join(tmp, "bot")) else tmp
    shutil.move(src, CODE_DIR)
else:
    raise SystemExit("Set REPO_URL or ZIP_PATH in the cell above.")
print("code:", sorted(os.listdir(CODE_DIR))[:12])

In [ ]:
# System packages (Tesseract Hindi + English, Pango for WeasyPrint, cloudflared for the Web App) + Python packages
!apt-get -qq update > /dev/null
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-hin libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz-subset0 fonts-noto-core wget > /dev/null
!curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!python -m pip install -q -r /content/rvsh/requirements.txt
!python --version && tesseract --list-langs 2>/dev/null | tail -n +2 | tr '\n' ' ' && cloudflared --version

In [ ]:
# Settings from 🔑 Secrets → environment (never printed)
from google.colab import userdata

KEYS = ["BOT_TOKEN", "API_ID", "API_HASH", "OWNER_ID", "DUMP_CHANNEL_ID", "IMAGE_DUMP_CHANNEL_ID", "BRANDING",
        "IMAGE_BOT_TOKENS", "MONGO_URL", "QUIZ_DUMP_CHANNEL_ID", "FORCE_SUB_CHANNELS", "HTTP_PROXY",
        "CLOUDFLARED_TOKEN", "WEBAPP_BASE_URL"]
env = dict(os.environ)
missing = []
for k in KEYS:
    try:
        v = userdata.get(k)
    except Exception:
        v = None
    if v:
        env[k] = str(v)
    elif k in KEYS[:9]:
        missing.append(k)
env.update(RUNTIME_PROFILE="colab", RVSH_SUPERVISED="1", PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8",
           DOWNLOAD_ROOT=f"{DATA}/downloads", ENV_FILE=f"{DATA}/.env")
open(env["ENV_FILE"], "a").close()  # /bot → config.env writes here
print("missing secrets:", ", ".join(missing) or "none")

In [ ]:
# Run the bot (⏹ stops it gracefully). /restart restarts inside; /update restarts here on the new code.
import signal, time

crashes = 0
while True:
    p = subprocess.Popen([sys.executable, "-m", "bot"], cwd=CODE_DIR, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:  # the bot's console output, shown here
            print(line, end="")
        code = p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        try:
            p.wait(120)
        except subprocess.TimeoutExpired:
            p.kill()
        print("stopped")
        break
    print("bot exited with", code)
    if code == 3:          # /update: start the new code
        continue
    if code == 1 and crashes < 3:
        crashes += 1
        time.sleep(20)
        continue
    break                  # 0 = stopped / handed over · 2 = a setting is wrong (see above)

**Logs:** `/content/rvsh-data/downloads/logs/bot.log` (Files panel), or `/log` in the bot.
**Web App:** the free quick tunnel works here (its address changes every session); a Mini App link needs a fixed
address — see `/qz → Web App → 📱 Mini App setup`.